<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/01_signal_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Signal Construction

This notebook builds the project's first alternative-data signal: a **congressional
cluster-buy score**, and the **enrichment layer** that makes its smartest features
real. The research question it serves is whether clustered, conviction-weighted
congressional buying identifies equities with informational edge ahead of the
broader market.

The signal is the worked template; the remaining signals (government contracts,
lobbying, off-exchange) follow the same `BaseSignal` contract and join a weighted
composite. Notebook `02` tests whether the scores predict forward returns and
calibrates the feature weights against that evidence.

### Signal design

The score blends features, each encoding a testable hypothesis about what makes a
disclosed purchase informative:

| Feature | Hypothesis | Data source |
|---|---|---|
| `cluster` | More distinct members buying the same name carries more signal | Quiver |
| `size_vs_networth` | Larger trades signal conviction — measured *relative to the member's net worth* to remove the wealth confound | Quiver + net-worth reference |
| `committee_alignment` | A member whose committee oversees the traded sector may be better informed | committee reference |
| `recency` | More recent disclosures are more actionable | Quiver |
| `bipartisan` | Agreement across parties strengthens the read | Quiver |

Two of these features draw on data the Quiver client does not return — net worth
and committee membership — so the project sources them itself in an **enrichment
layer** (built below). A discipline runs through the whole signal: scores are keyed
on the **disclosure date**, never the trade date, since a trade becomes actionable
only once it is public.


## Data and setup

The pipeline defaults to **mock trades**, so this notebook runs with no Quiver key.
To use live Quiver data, store your token in Colab Secrets as `QUIVER_API_KEY` and
set `USE_LIVE = True`. The enrichment layer pulls public committee data either way,
so its real features are demonstrated even in mock mode.


In [1]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True once your token is saved in Secrets

Working in: /content/quant-equity-research


Install in editable mode and put the package on the session path. The added
`src` path entry lets the kernel import the freshly written modules — including the
new `enrichment` subpackage — without a restart.


In [2]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import quant_research
importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.2.0


## The enrichment layer

Net worth and committee membership are the two features the Quiver client cannot
supply, so the project sources them independently.

- **Committee membership** comes from the `@unitedstates/congress-legislators`
  project — an authoritative open dataset. A disclosed trade carries a member's
  name; the layer resolves that name to a bioguide id (handling middle names and
  suffixes), then to the member's current committees, and maps each committee to
  the sectors it oversees.
- **Net worth** comes from a curated reference table (`reference_data/networth.csv`),
  seeded with approximate public estimates and meant to be expanded from primary
  financial disclosures.
- **Ticker sector** comes from a static map, with a yfinance fallback for names not
  yet mapped.

Members the data cannot match, and tickers without a sector, degrade quietly — the
feature declines to fire rather than penalizing the name. Building `LiveEnrichment`
fetches the committee data once and caches it locally.


In [3]:
from quant_research.enrichment.live import LiveEnrichment

live = LiveEnrichment()   # one-time fetch of public committee data, cached to data/reference

import pandas as pd
demo = []
for name in ["Tommy Tuberville", "Dan Crenshaw", "Michael McCaul", "Ro Khanna"]:
    demo.append({
        "member": name,
        "bioguide": live.resolve(name),
        "net_worth": live.net_worth(name),
        "committee_sectors": ", ".join(sorted(live.committee_sectors(name))) or "—",
    })
pd.DataFrame(demo)

,member,bioguide,net_worth,committee_sectors
0,Tommy Tuberville,T000278,6000000.0,"Consumer Staples, Defense, Health Care"
1,Dan Crenshaw,C001120,3000000.0,"Consumer Discretionary, Energy, Technology, Ut..."
2,Michael McCaul,M001157,100000000.0,Defense
3,Ro Khanna,K000389,30000000.0,Defense


A quick alignment check confirms the join is doing real work: an Energy-committee
member aligns with an energy name, an Armed Services member with a defense name.


In [4]:
print("Crenshaw (Energy committee)  + XOM [Energy]   ->", live.is_aligned("Dan Crenshaw", "XOM"))
print("Tuberville (Armed Services)  + LMT [Defense]  ->", live.is_aligned("Tommy Tuberville", "LMT"))
print("Crenshaw                     + JPM [Financials]->", live.is_aligned("Dan Crenshaw", "JPM"))

Crenshaw (Energy committee)  + XOM [Energy]   -> True
Tuberville (Armed Services)  + LMT [Defense]  -> True
Crenshaw                     + JPM [Financials]-> False


## Construct the signal

We compute the signal as of today, passing the **live enrichment** so the net-worth
and committee features run on real data — even on mock trades, since the mock member
names are real. `compute` returns a DataFrame indexed by ticker, sorted by score,
carrying both the final `score` and the features behind it, so every score is fully
explainable.


In [5]:
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.congress import CongressSignal

client = QuiverClient(token=QUIVER_TOKEN) if USE_LIVE else QuiverClient(mock=True)
signal = CongressSignal(client, lookback_days=30, enrichment=live)

scores = signal.compute()
print(f"{len(scores)} tickers scored | trades: {'live' if USE_LIVE else 'mock'} | enrichment: live")
scores[["score", "n_buys", "cluster", "committee_alignment", "bipartisan"]].head(12)

https://api.quiverquant.com/beta/live/congresstrading
96 tickers scored | trades: live | enrichment: live


,score,n_buys,cluster,committee_alignment,bipartisan
ticker,,,,,
MSFT,100.0,5.0,3.0,0.0,0
INTC,51.8,1.0,1.0,0.0,0
AMD,34.7,3.0,2.0,0.0,0
T,33.3,2.0,2.0,0.0,0
FN,32.2,2.0,2.0,0.0,0
BRK.B,30.0,4.0,1.0,0.0,0
UBER,14.6,1.0,1.0,0.0,0
HUBB,10.5,5.0,1.0,0.0,0
TDG,10.1,5.0,1.0,0.0,0


### Reading the output

`score` is the final 0–100 conviction reading. Beside it sit the raw features, and
the `*_n` columns below are those features normalized across the cross-section — the
actual inputs to the weighted score.


In [6]:
comp_cols = [c for c in scores.columns if c.endswith("_n")]
scores[["score"] + comp_cols].head(8)

,score,cluster_n,size_vs_networth_n,committee_alignment_n,recency_n,bipartisan_n
ticker,,,,,,
MSFT,100.0,1.0,0.404,0.0,0.703,0.0
INTC,51.8,0.0,1.000,0.0,0.081,0.0
AMD,34.7,0.5,0.005,0.0,0.162,0.0
T,33.3,0.5,0.025,0.0,0.081,0.0
FN,32.2,0.5,0.003,0.0,0.081,0.0
BRK.B,30.0,0.0,0.008,0.0,1.000,0.0
UBER,14.6,0.0,0.248,0.0,0.081,0.0
HUBB,10.5,0.0,0.019,0.0,0.324,0.0


### Why the leaders score where they do

A score is only useful if it is explainable. The chart shades each candidate by
committee alignment, so the contribution of that feature is visible at a glance,
alongside the cluster and conviction components.


In [7]:
import plotly.express as px

top = scores.head(15).reset_index()
fig = px.bar(top, x="score", y="ticker", orientation="h",
             color="committee_alignment_n", color_continuous_scale="Tealgrn",
             labels={"committee_alignment_n": "committee\nalignment"},
             title="Congress signal — top 15 (shading = committee alignment)")
fig.update_layout(yaxis=dict(autorange="reversed"), height=460)
fig.show()

## The disclosure-lag discipline, in practice

The signal keys on the disclosure date rather than the trade date. Quantifying the
gap makes the look-ahead concern concrete: a backtest scoring these trades on their
*transaction* date would be acting on information not yet public.


In [8]:
raw = client.congress_trades()
lag = (raw.report_date - raw.transaction_date).dt.days
print("Disclosure lag (days) — min / mean / max:",
      int(lag.min()), "/", round(lag.mean(), 1), "/", int(lag.max()))
print("Share disclosed more than 30 days after the trade:", f"{(lag > 30).mean():.0%}")

https://api.quiverquant.com/beta/live/congresstrading
Disclosure lag (days) — min / mean / max: 1 / 22.3 / 80
Share disclosed more than 30 days after the trade: 16%


## What notebook 02 will settle

The feature weights here are **priors**, not conclusions. The next notebook runs an
event study: for each disclosure it measures forward returns at several horizons,
then buckets those returns by feature — large committee-aligned buys in one bucket,
small unaligned ones in another. A feature that genuinely separates winners from
losers earns its weight from the data; one that does not is dropped to guard against
overfitting.


## Recap and next

The first signal is built and genuinely complete: a transparent congressional
cluster-buy score whose conviction feature normalizes against real net worth and
whose alignment feature draws on real committee assignments, all respecting
disclosure timing. The enrichment layer that powers it is reusable by every signal
that follows.

Next, the remaining Hobbyist-tier signals — government contracts, lobbying, and
off-exchange — adopt the same `BaseSignal` contract, and a weighted composite blends
them into one ranking. Then notebook `02` runs the event study that calibrates these
weights against real forward returns.
